# Fetch and Process Datasets

**Purpose:**
- Demonstrate where raw data comes from
- Call existing fetch and process modules
- Verify schemas used by demo notebooks

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
if _cwd.name != "notebooks":
    sys.path.insert(0, str(_cwd / "notebooks"))

from utils import setup_root
ROOT = setup_root()

## Configuration

In [ ]:
DATASETS = ["gsm8k", "qmsum"]
SEED = 42

## Dataset Locations

In [ ]:
RAW_DIR = ROOT / "data/raw"
PROCESSED_DIR = ROOT / "data/processed"
EVAL_DIR = ROOT / "data/eval"

print("Expected directories:")
print(f"RAW: {RAW_DIR}")
print(f"PROCESSED: {PROCESSED_DIR}")
print(f"EVAL: {EVAL_DIR}")

## Fetch and Process

In [ ]:
import subprocess
import sys

for ds in DATASETS:
    raw_path = RAW_DIR / ds
    filename = "gsm8k_100.jsonl" if ds == "gsm8k" else "qmsum_meeting_qa_100.jsonl"
    eval_path = EVAL_DIR / filename
    
    print(f"Fetching {ds}...")
    subprocess.run([sys.executable, "scripts/fetch_dataset.py", "--dataset", ds, "--stage", "raw"], check=True)
    
    print(f"Processing {ds}...")
    subprocess.run([sys.executable, "scripts/fetch_dataset.py", "--dataset", ds, "--stage", "processed"], check=True)
    subprocess.run([sys.executable, "scripts/fetch_dataset.py", "--dataset", ds, "--stage", "eval"], check=True)

## Audit and Preview

In [ ]:
import json

for ds in DATASETS:
    filename = "gsm8k_100.jsonl" if ds == "gsm8k" else "qmsum_meeting_qa_100.jsonl"
    eval_path = EVAL_DIR / filename
    if not eval_path.exists():
        continue
        
    with open(eval_path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
        
    print(f"\n--- {ds.upper()} ---")
    print(f"Total evaluation rows: {len(rows)}")
    if rows:
        sample = rows[0]
        print(f"Schema keys: {list(sample.keys())}")
        print("Sample preview:")
        print(json.dumps(sample, indent=2)[:500] + "...")